# SPG-Aware Retrieval Methods Testing

This notebook tests all the retrieval methods for the Semantic Property Graph (SPG) financial knowledge graph.

## Available Methods

1. **vector_similarity_search** - Basic vector search with full properties
2. **typed_vector_search** - Type-filtered vector search (SPG-aware)
3. **entity_graph_search** - Entity relationship search with auto-expansion
4. **semantic_path_search** - Multi-hop path discovery
5. **question_aware_subgraph_retrieval** - Intelligent question-based subgraph extraction

## 1. Setup and Imports

In [1]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python path: {sys.path[0]}")

Project root: /Users/duykhangh/Work/HCMUT/thesis-llms-multilayer-graph
Python path: /Users/duykhangh/Work/HCMUT/thesis-llms-multilayer-graph


In [2]:
# Load environment variables
from dotenv import load_dotenv

env_file = project_root / ".env"
if env_file.exists():
    load_dotenv(env_file)
    print("✓ Environment variables loaded")
else:
    print("⚠ No .env file found")

# Verify required environment variables
required_vars = ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "GOOGLE_API_KEY"]
missing = [var for var in required_vars if not os.getenv(var)]

if missing:
    print(f"❌ Missing environment variables: {missing}")
else:
    print(f"✓ All required environment variables present")
    print(f"  - Neo4j URI: {os.getenv('NEO4J_URI')}")
    print(f"  - Neo4j Database: {os.getenv('NEO4J_DATABASE', 'neo4j')}")
    print(f"  - Embedding Model: {os.getenv('EMBEDDING_MODEL', 'gemini-embedding-001')}")

✓ Environment variables loaded
✓ All required environment variables present
  - Neo4j URI: bolt://localhost:7687
  - Neo4j Database: financebench1
  - Embedding Model: gemini-embedding-001


In [ ]:
# Import retriever
from rag.retrievers import Neo4jRetriever

print("✓ Imports successful")

In [ ]:
# Import preprocessing utilities
from rag.state import ChatState
from langchain_core.messages import HumanMessage

print("✓ Imports successful")

## 2. Initialize Retriever

In [5]:
# Create retriever instance
try:
    retriever = Neo4jRetriever()
    print("✓ Neo4j Retriever initialized successfully")
    print(f"  - Database: {retriever.database}")
    print(f"  - Embedding model: {retriever.embedding_model}")
    print(f"  - Embedding type: {retriever.embedding_type}")
except Exception as e:
    print(f"❌ Failed to initialize retriever: {e}")
    raise

[Neo4j Retriever] Using Gemini embedding model: gemini-embedding-001
[Neo4j Retriever] Embedding dimension: 3072
✓ Neo4j Retriever initialized successfully
  - Database: financebench1
  - Embedding model: gemini-embedding-001
  - Embedding type: gemini


## 3. Test Database Connection

In [6]:
# Test basic Neo4j connectivity
with retriever.driver.session(database=retriever.database) as session:
    # Count nodes
    result = session.run("MATCH (n) RETURN count(n) as node_count")
    node_count = result.single()["node_count"]
    
    # Count relationships
    result = session.run("MATCH ()-[r]->() RETURN count(r) as rel_count")
    rel_count = result.single()["rel_count"]
    
    # Get node labels
    result = session.run("CALL db.labels()")
    labels = [record["label"] for record in result]
    
    # Get relationship types
    result = session.run("CALL db.relationshipTypes()")
    rel_types = [record["relationshipType"] for record in result]
    
    print(f"✓ Connected to Neo4j database: {retriever.database}")
    print(f"\nGraph Statistics:")
    print(f"  - Total nodes: {node_count:,}")
    print(f"  - Total relationships: {rel_count:,}")
    print(f"  - Node labels ({len(labels)}): {', '.join(labels[:10])}")
    print(f"  - Relationship types ({len(rel_types)}): {', '.join(rel_types[:10])}")

✓ Connected to Neo4j database: financebench1

Graph Statistics:
  - Total nodes: 395
  - Total relationships: 580
  - Node labels (14): Chunk, Entity, Company, FinancialEvent, RegulatoryFiling, FinancialStatement, Industry, BusinessSegment, EventCategory, Product
  - Relationship types (62): SOURCE, REPORTED_FINANCIALS, isA, FILED, belongTo, ENTERS_INTO, PRODUCES, OWNS, HAS_BUSINESS_SEGMENT, HAS_SUBSIDIARY


## 4. Helper Functions

In [7]:
import os
import json
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

def preprocess_query(question: str):
    """
    LLM-based query preprocessing to extract:
    1. Mentioned entities (explicitly stated in question)
    2. Source entities (where to find the information in the graph)
    3. Entity types needed
    4. Relationship types needed
    5. Query intent
    
    Args:
        question: Natural language question
        
    Returns:
        dict with: mentioned_entities, source_entities, entity_types, relationship_types, intent, reasoning
    """
    # Initialize LLM
    base_url = os.getenv("CHAT_OPENAI_BASE_URL")
    api_key = os.getenv("OPENAI_API_KEY") or os.getenv("GOOGLE_API_KEY")
    
    if base_url and not api_key:
        api_key = "EMPTY"
    
    llm = ChatOpenAI(
        model=os.getenv("CHAT_LLM_MODEL", "gpt-4o-mini"),
        temperature=0.1,
        api_key=api_key,
        base_url=base_url if base_url else None
    )
    
    system_prompt = """You are an expert at analyzing financial questions and determining where to find information in a knowledge graph.

**Your task**: Extract two types of entities:
1. **Mentioned Entities**: Entities explicitly mentioned in the question
2. **Source Entities**: Entities where the answer can be found (even if not mentioned)

**Example - Key Distinction:**
Question: "What was the revenue in Q4 2023?"
- Mentioned: ["Q4 2023", "revenue"]
- Source: ["Company", "FinancialStatement"] ← These contain revenue data even though not mentioned!

**SPG Schema - Entity Types:**
- Company, Executive, Product, BusinessSegment, GeographicRegion
- FinancialStatement, FinancialEvent, RegulatoryFiling, MarketData
- Industry, TimePeriod

**SPG Schema - Relationship Types:**
- REPORTED_FINANCIALS, OWNS, EMPLOYS, PRODUCES, OPERATES_IN
- HAS_SEGMENT, FILED, INVOLVED_IN, HAS_MARKET_DATA
- COMPETES_WITH, WORKS_FOR, LEADS, BELONGS_TO

**Output Format (JSON):**
{
  "mentioned_entities": ["Entities explicitly stated in question"],
  "source_entities": ["Entity types/names where answer is found"],
  "entity_types": ["Entity types needed for search"],
  "relationship_types": ["Relationship types needed"],
  "intent": "factual_lookup|comparison|explanation|retrieval|relationship_query|aggregation",
  "reasoning": "Explanation: what's mentioned vs where to find it"
}

**Guidelines:**
1. **Mentioned entities**: Extract exactly what's in the question
2. **Source entities**: Identify WHERE this information lives in the graph
   - Specific names if known: ["Apple", "FinancialStatement"]
   - Generic types if not: ["Company", "FinancialStatement"]
3. Be generous with source_entities - include all places the answer might be

**Examples:**

Q: "What was Apple's revenue in Q4 2023?"
{
  "mentioned_entities": ["Apple", "Q4 2023", "revenue"],
  "source_entities": ["Apple", "FinancialStatement"],
  "entity_types": ["Company", "FinancialStatement", "TimePeriod"],
  "relationship_types": ["REPORTED_FINANCIALS"],
  "intent": "factual_lookup",
  "reasoning": "Mentioned: Apple, Q4 2023, revenue. Source: Apple entity and its FinancialStatement nodes via REPORTED_FINANCIALS."
}

Q: "What was the revenue in Q4 2023?"
{
  "mentioned_entities": ["Q4 2023", "revenue"],
  "source_entities": ["Company", "FinancialStatement"],
  "entity_types": ["Company", "FinancialStatement", "TimePeriod"],
  "relationship_types": ["REPORTED_FINANCIALS"],
  "intent": "factual_lookup",
  "reasoning": "Mentioned: Q4 2023, revenue. No company specified, so search any Company with FinancialStatement data."
}

Q: "Who is the CEO of Microsoft?"
{
  "mentioned_entities": ["Microsoft", "CEO"],
  "source_entities": ["Microsoft", "Executive"],
  "entity_types": ["Company", "Executive"],
  "relationship_types": ["EMPLOYS"],
  "intent": "factual_lookup",
  "reasoning": "Mentioned: Microsoft, CEO. Source: Microsoft entity and Executive entities via EMPLOYS."
}

Q: "Compare Google and Facebook's profit margins"
{
  "mentioned_entities": ["Google", "Facebook", "profit margins"],
  "source_entities": ["Google", "Facebook", "FinancialStatement"],
  "entity_types": ["Company", "FinancialStatement"],
  "relationship_types": ["REPORTED_FINANCIALS"],
  "intent": "comparison",
  "reasoning": "Mentioned: Google, Facebook, profit margins. Source: Both companies and their FinancialStatement nodes."
}

Now extract from the user's question:"""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Question: {question}")
    ]
    
    response = llm.invoke(messages)
    
    # Parse JSON response
    result_text = response.content.strip()
    
    # Extract JSON from markdown code blocks if present
    if "```json" in result_text:
        result_text = result_text.split("```json")[1].split("```")[0].strip()
    elif "```" in result_text:
        result_text = result_text.split("```")[1].split("```")[0].strip()
    
    result = json.loads(result_text)
    result['original_question'] = question
    
    return result

print("✓ LLM-based query preprocessing function defined (with source entity detection)")

✓ LLM-based query preprocessing function defined (with source entity detection)


In [9]:
def print_result(title, result):
    """Pretty print retrieval results"""
    print("\n" + "="*80)
    print(f"TEST: {title}")
    print("="*80)
    print(result)
    print("="*80 + "\n")

def truncate_text(text, max_length=200):
    """Truncate long text for display"""
    if len(text) > max_length:
        return text[:max_length] + "..."
    return text

## 5. Test: Basic Vector Similarity Search

In [10]:
# Test basic vector search with full properties
query = "3M CAPEX and Fixed Assets"
result = retriever.vector_similarity_search(
    query=query,
    limit=3,
    similarity_threshold=0.5
)

print_result("Basic Vector Similarity Search", result)


TEST: Basic Vector Similarity Search

**Capital Expenditures and Investments** (FinancialStatement) [similarity: 0.677]
  - ID: fs_capital_expenditures_and_investments_2022
  - Description: A section detailing a company's spending on fixed assets and other long-term investments for the year ended December 31, 2022, such as wireless networks and advanced technologies.
  - Source Chunks (Text Evidence):
    [Chunk 1, Page 22]: "Corporate and other also includes the historical results of divested \nbusinesses, including Verizon Media Group (Verizon Media) divested in September 2021, and other adjustments and gains and \nlosses that are not allocated in assessing segment performance due to their nature Although such transactions are excluded \nfrom the business segment results, they are included in reported consolidated earnings Gains and losses from these \ntransactions that are not individually significant are included in segment results as these items are included in the chief operatin

## 6. Test: Type-Filtered Vector Search (SPG-Aware)

In [ ]:
# Test 1: Filter by entity type only
query = "financial earnings revenue"
result = retriever.typed_vector_search(
    query=query,
    entity_types=["Company"],
    relationship_types=None,
    limit=3,
    similarity_threshold=0.5
)

print_result("Type-Filtered Search: Companies Only", result)

In [ ]:
# Test 2: Filter by entity type AND relationship type
query = "revenue profit financial metrics"
result = retriever.typed_vector_search(
    query=query,
    entity_types=["Company", "FinancialStatement"],
    relationship_types=["REPORTED_FINANCIALS"],
    limit=3,
    similarity_threshold=0.5
)

print_result("Type-Filtered Search: Companies with REPORTED_FINANCIALS", result)

In [ ]:
# Test 3: Filter by multiple relationship types
query = "ownership subsidiary structure"
result = retriever.typed_vector_search(
    query=query,
    entity_types=["Company"],
    relationship_types=["OWNS", "COMPETES_WITH"],
    limit=3,
    similarity_threshold=0.4
)

print_result("Type-Filtered Search: Ownership Relationships", result)

## 7. Test: Entity Graph Search with Auto-Expansion

In [ ]:
# Test 1: Search for relationships between specific entities
# This will auto-expand if no direct relationships found
entities = "Company, FinancialStatement"
result = retriever.entity_graph_search(
    entities=entities,
    depth=1  # Depth parameter is unused, kept for compatibility
)

print_result("Entity Graph Search: Company and FinancialStatement", result)

In [ ]:
# Test 2: Search with entities that might need expansion
entities = "revenue, profit, Company"
result = retriever.entity_graph_search(
    entities=entities,
    depth=1
)

print_result("Entity Graph Search: With Auto-Expansion", result)

## 8. Test: Semantic Path Search (Multi-Hop)

In [15]:
state = preprocess_query("Does Adobe have an improving operating margin profile as of FY2022? If operating margin is not a useful metric for a company like this, then state that and explain why.")
print(state)

{'mentioned_entities': ['Adobe', 'operating margin', 'FY2022'], 'source_entities': ['Adobe', 'FinancialStatement', 'Industry'], 'entity_types': ['Company', 'FinancialStatement', 'Industry', 'TimePeriod'], 'relationship_types': ['REPORTED_FINANCIALS', 'BELONGS_TO'], 'intent': 'factual_lookup|explanation', 'reasoning': "Mentioned: Adobe, operating margin, FY2022. Source: Adobe's FinancialStatement for operating margin data, and potentially Industry entity to assess if operating margin is a relevant metric for companies like Adobe. Explanation may require industry context.", 'original_question': 'Does Adobe have an improving operating margin profile as of FY2022? If operating margin is not a useful metric for a company like this, then state that and explain why.'}


In [16]:
# Test 1: Find paths between two entities
# start_entity = "3M Company and Subsidiaries"
# end_entity = "FinancialStatement"
start_entities = state.get("mentioned_entities")
end_entities = state.get("source_entities")
for start_entity in start_entities:
    for end_entity in end_entities:
        result = retriever.semantic_path_search(
            start_entity=start_entity,
            end_entity=end_entity,
            max_hops=3,
            path_relationship_types=None,
            limit_paths=5
        )

        print_result(f"Semantic Path Search: {start_entity} → {end_entity}", result)


[Semantic Path Search] Finding paths: 'Adobe' → 'Adobe'
  Max hops: 3
  Found 3 start entities:
    - ADOBE INC. [Entity, Company] (similarity: 0.900, match: partial)
    - CONSOLIDATED STATEMENTS OF INCOME [Entity, FinancialStatement] (similarity: 0.900, match: partial)
    - CONSOLIDATED STATEMENTS OF CASH FLOWS [Entity, FinancialStatement] (similarity: 0.900, match: partial)
  Found 3 end entities:
    - ADOBE INC. [Entity, Company] (similarity: 0.900, match: partial)
    - CONSOLIDATED STATEMENTS OF INCOME [Entity, FinancialStatement] (similarity: 0.900, match: partial)
    - CONSOLIDATED STATEMENTS OF CASH FLOWS [Entity, FinancialStatement] (similarity: 0.900, match: partial)
  ✓ Found 16 paths


TEST: Semantic Path Search: Adobe → Adobe

## Semantic Path Search: 'Adobe' → 'Adobe'
**Found 16 paths** (max 3 hops)


### Path 1 (length: 1, score: 1.300)
**Start**: ADOBE INC. [Entity, Company] (sim: 0.900)
**End**: CONSOLIDATED STATEMENTS OF INCOME [Entity, FinancialStatement] (sim: 

In [19]:
# Test 2: Find paths with relationship type filtering
start_entity = "Company"
end_entity = "revenue"
result = retriever.semantic_path_search(
    start_entity=start_entity,
    end_entity=end_entity,
    max_hops=3,
    path_relationship_types=["REPORTED_FINANCIALS"],
    limit_paths=3
)

print_result(f"Semantic Path Search: {start_entity} → {end_entity} (REPORTED_FINANCIALS only)", result)


[Semantic Path Search] Finding paths: 'Company' → 'revenue'
  Max hops: 3
  Relationship type filter: ['REPORTED_FINANCIALS']
  Found 3 start entities:
    - The Company [Entity, Company] (similarity: 0.630)
    - BioNTech [Entity, Company] (similarity: 0.611)
    - Balance Sheet [Entity, FinancialStatement] (similarity: 0.610)
  Found 3 end entities:
    - U.S [Entity, GeographicRegion] (similarity: 0.660)
    - Consolidated Operating Revenues [Entity, FinancialStatement] (similarity: 0.646)
    - U.S. government [Chunk, Entity] (similarity: 0.642)

TEST: Semantic Path Search: Company → revenue (REPORTED_FINANCIALS only)
No paths found between 'Company' and 'revenue' within 3 hops



## 9. Test: Question-Aware Subgraph Retrieval (Most Intelligent)

In [ ]:
# Test 1: Financial metric question
question = "What is the company's revenue and profit?"
result = retriever.question_aware_subgraph_retrieval(
    question=question,
    entities=None,  # Auto-extract
    max_hops=2,
    include_context=True,
    property_filters=None
)

print_result("Question-Aware Subgraph: Financial Metrics", result)

In [ ]:
# Test 2: Ownership question
question = "What companies does the parent company own?"
result = retriever.question_aware_subgraph_retrieval(
    question=question,
    max_hops=2,
    include_context=True
)

print_result("Question-Aware Subgraph: Ownership", result)

In [ ]:
# Test 3: With property filters
question = "Show me financial data from Q4 2023"
result = retriever.question_aware_subgraph_retrieval(
    question=question,
    max_hops=2,
    include_context=True,
    property_filters={"period": "Q4 2023"}
)

print_result("Question-Aware Subgraph: With Property Filters", result)

## 10. Compare All Methods

In [ ]:
import time

test_query = "company revenue financial data"

print("\n" + "="*80)
print("COMPARISON: All Retrieval Methods")
print("="*80 + "\n")

# 1. Basic vector search
start = time.time()
result1 = retriever.vector_similarity_search(test_query, limit=3)
time1 = time.time() - start
print(f"1. Vector Similarity Search: {time1:.2f}s")
print(f"   Result length: {len(result1)} characters\n")

# 2. Typed vector search
start = time.time()
result2 = retriever.typed_vector_search(
    test_query, 
    entity_types=["Company"], 
    limit=3
)
time2 = time.time() - start
print(f"2. Typed Vector Search: {time2:.2f}s")
print(f"   Result length: {len(result2)} characters\n")

# 3. Entity graph search
start = time.time()
result3 = retriever.entity_graph_search("Company, FinancialStatement")
time3 = time.time() - start
print(f"3. Entity Graph Search: {time3:.2f}s")
print(f"   Result length: {len(result3)} characters\n")

# 4. Question-aware subgraph
start = time.time()
result4 = retriever.question_aware_subgraph_retrieval(
    "What is the company's revenue?",
    max_hops=2
)
time4 = time.time() - start
print(f"4. Question-Aware Subgraph: {time4:.2f}s")
print(f"   Result length: {len(result4)} characters\n")

print("="*80)

## 11. Test with Real Financial Queries

In [8]:
# Get a sample company name from the database
with retriever.driver.session(database=retriever.database) as session:
    result = session.run("""
        MATCH (c:Company)
        WHERE c.name IS NOT NULL
        RETURN c.name as company_name
        LIMIT 5
    """)
    companies = [record["company_name"] for record in result]
    
    print("Sample companies in database:")
    for i, company in enumerate(companies, 1):
        print(f"  {i}. {company}")

Sample companies in database:
  1. Verizon Media Group
  2. Corning Incorporated and Subsidiary Companies
  3. JPMorgan Chase
  4. JPMorgan Chase Bank, N.A
  5. J.P Morgan Securities LLC


In [10]:
# Test with real company (replace with actual company name from above)
if companies:
    company_name = companies[0]
    
    # Test 1: Revenue question
    # question = f"What was {company_name}'s revenue?"
    question = "3M CAPEX and Fixed Assets"
    result = retriever.question_aware_subgraph_retrieval(
        question=question,
        max_hops=2
    )
    print_result(f"Real Query: {question}", result)
    
    # Test 2: Financial relationships
    result = retriever.typed_vector_search(
        query=company_name,
        entity_types=["Company"],
        relationship_types=["REPORTED_FINANCIALS"],
        limit=3
    )
    print_result(f"Real Query: {company_name} Financial Relationships", result)
else:
    print("⚠ No companies found in database")


[Subgraph Retrieval] Question: 3M CAPEX and Fixed Assets
  Stage 1: Found 5 seed entities
    - Capital Expenditures and Investments [Entity, FinancialStatement] (sim: 0.677)
    - Consolidated Balance Sheets [Entity, FinancialStatement] (sim: 0.631)
    - CONSOLIDATED BALANCE SHEETS [Entity, FinancialStatement] (sim: 0.628)
  Stage 2: Inferred relationship types: ALL
  Stage 3: Extracted subgraph - 59 nodes, 67 edges

TEST: Real Query: 3M CAPEX and Fixed Assets

## Question-Specific Subgraph
**Question**: 3M CAPEX and Fixed Assets
**Subgraph Size**: 59 entities, 67 relationships

### Entities in Subgraph:

1. **Capital Expenditures and Investments** [Entity, FinancialStatement]
   Description: A section detailing a company's spending on fixed assets and other long-term investments for the year ended December 31, 2022, such as wireless networks and advanced technologies.
   Source Evidence: "Corporate and other also includes the historical results of divested \nbusinesses, including V

## 12. Cleanup

In [ ]:
# Close Neo4j connection
retriever.driver.close()
print("✓ Neo4j connection closed")

## Summary

This notebook tested all 5 SPG-aware retrieval methods:

1. ✅ **vector_similarity_search** - Basic semantic search with full properties
2. ✅ **typed_vector_search** - Type-filtered search (Company, REPORTED_FINANCIALS, etc.)
3. ✅ **entity_graph_search** - Relationship search with auto-expansion
4. ✅ **semantic_path_search** - Multi-hop path discovery between entities
5. ✅ **question_aware_subgraph_retrieval** - Intelligent question-based retrieval

### Method Selection Guide:

- **Simple semantic search**: Use `vector_similarity_search`
- **Type-specific search**: Use `typed_vector_search` with entity/relationship filters
- **Known entities**: Use `entity_graph_search` (auto-expands if needed)
- **Connection discovery**: Use `semantic_path_search` for "how is X related to Y"
- **Complex questions**: Use `question_aware_subgraph_retrieval` for comprehensive context

All methods return **complete SPG relationship properties** including financial metrics, temporal data, and contextual information!